In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,udf
from pyspark.sql.types import StringType,IntegerType
import time

In [3]:
spark=SparkSession.builder.appName("sparkapp").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 07:20:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.range(0, 100000).withColumn("value", col("id"))

In [5]:
df.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [6]:
print("Before partitions:",df.rdd.getNumPartitions())

Before partitions: 2


In [7]:
df_repartitioned = df.repartition(10)

In [8]:
print("After partitions:", df_repartitioned.rdd.getNumPartitions())

[Stage 1:>                                                          (0 + 2) / 2]

After partitions: 10


In [9]:
df_coalesced=df_repartitioned.coalesce(2)

In [10]:
print("After coalesced:",df_coalesced.rdd.getNumPartitions())

After coalesced: 2


26/06/14 07:20:42 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [11]:
df_optimized=df.filter(col("value") >500).filter(col("id")<50000).select("id","value")

In [12]:
df_optimized.explain()

== Physical Plan ==
*(1) Project [id#0L, id#0L AS value#2L]
+- *(1) Filter ((id#0L > 500) AND (id#0L < 50000))
   +- *(1) Range (0, 100000, step=1, splits=2)




In [13]:
start_time =time.time()
count_uncached=df_optimized.count()
end_time=time.time()
print(f"count: {count_uncached} time: {round(end_time - start_time,4)} s")

count: 49499 time: 0.414 s


In [14]:
df_optimized.cache()

DataFrame[id: bigint, value: bigint]

In [15]:
start_time =time.time()
count_first_cached=df_optimized.count()
end_time=time.time()
print(f"count: {count_first_cached} time: {round(end_time - start_time,4)} s")

count: 49499 time: 0.8531 s


In [16]:
start_time =time.time()
count_second_cached=df_optimized.count()
end_time=time.time()
print(f"count: {count_second_cached} time: {round(end_time - start_time,4)} s")

count: 49499 time: 0.1761 s


In [17]:
start_time =time.time()
count_third_cached=df_optimized.count()
end_time=time.time()
print(f"count: {count_third_cached} time: {round(end_time - start_time,4)} s")

count: 49499 time: 0.1696 s


In [18]:
df_optimized.unpersist()

DataFrame[id: bigint, value: bigint]

In [19]:
data = [("alice", 30), ("akshya", 16)]
df = spark.createDataFrame(data, ["Name", "Age"])
df.show()

+------+---+
|  Name|Age|
+------+---+
| alice| 30|
|akshya| 16|
+------+---+



In [20]:
def can_drive(age):
    if age>=18:
        return "Can drive"
    return "Cannot drive"

In [21]:
age_categorize=udf(can_drive,StringType())

In [23]:
df_m = df.withColumn("category",age_categorize(col("Age")))

In [24]:
df_m.show()

+------+---+------------+
|  Name|Age|    category|
+------+---+------------+
| alice| 30|   Can drive|
|akshya| 16|Cannot drive|
+------+---+------------+



In [25]:
spark.udf.register("sql_categorize",can_drive,StringType())

<function __main__.can_drive(age)>

In [27]:
df.createOrReplaceTempView("people")

In [29]:
sql_df = spark.sql("""
 SELECT
 Name,
 Age,
 sql_categorize(Age) AS Category
 FROM people
""")


In [30]:
sql_df.show()

+------+---+------------+
|  Name|Age|    Category|
+------+---+------------+
| alice| 30|   Can drive|
|akshya| 16|Cannot drive|
+------+---+------------+



In [31]:
spark.stop()